# 31 - PF-012 a PF-019: agente endurecido y endpoints del servicio Python

Este notebook agrega reglas productivas del agente, reportes por caso, endpoints FastAPI, smoke tests y Dockerfile base.


In [1]:
from pathlib import Path
import json, sys, datetime
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PFI_ROOT = Path('/content/drive/MyDrive/PFI_MVP')
REPO_ROOT = PFI_ROOT / 'repo'
PACKAGE_ROOT = REPO_ROOT / 'ai_service' / 'pfi_ai_service'
AI_SERVICE_ROOT = REPO_ROOT / 'ai_service'
CONFIG_ROOT = REPO_ROOT / 'config'
RESULTS_ROOT = PFI_ROOT / 'results' / 'PF012_PF019_service_endpoints_agent'
DOCS_ROOT = REPO_ROOT / 'docs'
BACKLOG_ROOT = REPO_ROOT / 'backlogProducto'
for p in [PACKAGE_ROOT, AI_SERVICE_ROOT, CONFIG_ROOT, RESULTS_ROOT, DOCS_ROOT, BACKLOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
print('REPO_ROOT:', REPO_ROOT, '| exists:', REPO_ROOT.exists())


Mounted at /content/drive
REPO_ROOT: /content/drive/MyDrive/PFI_MVP/repo | exists: True


In [2]:
# Entradas esperadas
required = {
    'data_freeze_config': CONFIG_ROOT / 'data_freeze_config.json',
    'model_registry_final': CONFIG_ROOT / 'model_registry_final.json',
    'preprocessing': PACKAGE_ROOT / 'preprocessing.py',
    'measurements': PACKAGE_ROOT / 'measurements.py',
    'overlay': PACKAGE_ROOT / 'overlay.py',
    'inference': PACKAGE_ROOT / 'inference.py',
    'pipeline': PACKAGE_ROOT / 'pipeline.py',
}
for k,p in required.items():
    print(k, '->', p, '| exists:', p.exists())


data_freeze_config -> /content/drive/MyDrive/PFI_MVP/repo/config/data_freeze_config.json | exists: True
model_registry_final -> /content/drive/MyDrive/PFI_MVP/repo/config/model_registry_final.json | exists: True
preprocessing -> /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/preprocessing.py | exists: True
measurements -> /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/measurements.py | exists: True
overlay -> /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/overlay.py | exists: True
inference -> /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/inference.py | exists: True
pipeline -> /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/pipeline.py | exists: True


In [3]:
# Escritura de módulos PF012-PF019
files = {
    PACKAGE_ROOT / 'agent_policy.py': "from __future__ import annotations\nfrom dataclasses import dataclass, asdict\nfrom typing import Any, Dict, List\n\n@dataclass\nclass AgentDecision:\n    priority: str\n    status: str\n    flags: List[str]\n    reasons: List[str]\n    action_recommended: str\n    human_review_required: bool = True\n    not_clinical_diagnosis: bool = True\n    def to_dict(self) -> Dict[str, Any]:\n        return asdict(self)\n\ndef _f(v, default=0.0):\n    try:\n        return default if v is None else float(v)\n    except Exception:\n        return default\n\ndef evaluate_agent_policy(result: Dict[str, Any]) -> Dict[str, Any]:\n    plane = str(result.get('plane', 'unknown'))\n    foreground_ratio = _f(result.get('foreground_ratio'), 0.0)\n    mean_conf = _f(result.get('mean_fg_confidence', result.get('mean_confidence', 1.0)), 1.0)\n    present_classes = result.get('present_classes') or []\n    missing_expected = result.get('missing_expected_classes') or []\n    warnings = list(result.get('warnings') or [])\n    errors = list(result.get('errors') or [])\n    flags, reasons = [], []\n    if errors:\n        flags.append('error_pipeline'); reasons.append('El pipeline reportó errores técnicos durante la ejecución.')\n    if foreground_ratio <= 0.005:\n        flags.append('foreground_muy_bajo'); reasons.append('La proporción de máscara segmentada es extremadamente baja.')\n    elif foreground_ratio < 0.02:\n        flags.append('foreground_bajo'); reasons.append('La proporción de máscara segmentada es baja y requiere revisión.')\n    elif foreground_ratio > 0.55:\n        flags.append('foreground_muy_alto'); reasons.append('La proporción de máscara segmentada es inusualmente alta.')\n    if mean_conf < 0.70:\n        flags.append('confianza_baja'); reasons.append('La confianza media de foreground está por debajo del umbral esperado.')\n    elif mean_conf < 0.82:\n        flags.append('confianza_intermedia'); reasons.append('La confianza media de foreground es intermedia.')\n    if missing_expected:\n        flags.append('clases_esperadas_ausentes'); reasons.append('Faltan clases esperadas en la salida.')\n    if not present_classes:\n        flags.append('sin_clases_detectadas'); reasons.append('No se detectaron clases anatómicas útiles.')\n    if warnings:\n        flags.append('warnings_pipeline'); reasons.extend([f'Advertencia técnica: {w}' for w in warnings[:3]])\n    high = {'error_pipeline','foreground_muy_bajo','foreground_muy_alto','confianza_baja','sin_clases_detectadas'}\n    medium = {'foreground_bajo','confianza_intermedia','clases_esperadas_ausentes','warnings_pipeline'}\n    if any(f in high for f in flags):\n        priority, status = 'alta', 'requiere_revision_prioritaria'\n        action = 'Revisar antes de utilizar la salida como apoyo. Verificar imagen, máscara y clases detectadas.'\n    elif any(f in medium for f in flags):\n        priority, status = 'media', 'requiere_revision_con_atencion'\n        action = 'Revisar con atención la superposición y las mediciones antes de aceptar.'\n    else:\n        priority, status = 'baja', 'listo_para_revision_estandar'\n        action = 'Revisión profesional estándar antes de aceptar o descartar la salida.'\n        reasons.append('No se detectaron alertas técnicas relevantes según las reglas actuales.')\n    if plane not in {'sagittal','axial'}:\n        flags.append('plano_no_reconocido'); reasons.append('El plano informado no corresponde a sagittal o axial.')\n        if priority == 'baja': priority, status = 'media', 'requiere_revision_con_atencion'\n    return AgentDecision(priority, status, flags, reasons, action).to_dict()\n\ndef regression_cases():\n    return [\n        {'case_id':'REG_LOW','plane':'sagittal','foreground_ratio':0.10,'mean_fg_confidence':0.90,'present_classes':[1,2,3],'expected_priority':'baja'},\n        {'case_id':'REG_MED','plane':'axial','foreground_ratio':0.015,'mean_fg_confidence':0.88,'present_classes':[1,2],'expected_priority':'media'},\n        {'case_id':'REG_HIGH','plane':'sagittal','foreground_ratio':0.0,'mean_fg_confidence':0.20,'present_classes':[],'expected_priority':'alta'},\n    ]\n\ndef run_agent_regression():\n    rows=[]\n    for c in regression_cases():\n        d=evaluate_agent_policy(c); ok=d['priority']==c['expected_priority']\n        rows.append({'case_id':c['case_id'],'expected_priority':c['expected_priority'],'actual_priority':d['priority'],'ok':ok,'flags':d['flags']})\n    return {'total':len(rows),'passed':sum(1 for r in rows if r['ok']),'all_ok':all(r['ok'] for r in rows),'rows':rows}\n",
    PACKAGE_ROOT / 'case_reporting.py': 'from __future__ import annotations\nfrom pathlib import Path\nfrom typing import Any, Dict\nimport json, datetime\n\ndef now_iso():\n    return datetime.datetime.now().replace(microsecond=0).isoformat()\n\ndef build_case_report(request_payload: Dict[str, Any], technical_result: Dict[str, Any], agent_decision: Dict[str, Any]) -> Dict[str, Any]:\n    run_id = technical_result.get(\'run_id\') or request_payload.get(\'run_id\')\n    return {\n        \'run_id\': run_id,\n        \'created_at\': now_iso(),\n        \'case_id\': request_payload.get(\'case_id\', run_id),\n        \'plane\': technical_result.get(\'plane\', request_payload.get(\'plane\')),\n        \'model_key\': technical_result.get(\'model_key\', request_payload.get(\'model_key\')),\n        \'technical_result\': technical_result,\n        \'agent_decision\': agent_decision,\n        \'methodological_policy\': {\n            \'human_review_required\': True,\n            \'not_clinical_diagnosis\': True,\n            \'not_real_3d_reconstruction\': True,\n            \'professional_must_accept_correct_or_discard\': True,\n        },\n    }\n\ndef write_json_report(report: Dict[str, Any], output_dir: Path) -> Path:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    path = output_dir / f"{report.get(\'run_id\',\'unknown_run\')}.json"\n    path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    return path\n\ndef write_markdown_report(report: Dict[str, Any], output_dir: Path) -> Path:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    run_id = report.get(\'run_id\',\'unknown_run\')\n    tech = report.get(\'technical_result\', {})\n    dec = report.get(\'agent_decision\', {})\n    lines = [\n        f\'# Reporte técnico de caso - {run_id}\', \'\',\n        f"- case_id: `{report.get(\'case_id\')}`", f"- plane: `{report.get(\'plane\')}`", f"- model_key: `{report.get(\'model_key\')}`", \'\',\n        \'## Resultado técnico\', \'\',\n        f"- foreground_ratio: `{tech.get(\'foreground_ratio\')}`",\n        f"- present_classes: `{tech.get(\'present_classes\')}`",\n        f"- overlay_path: `{tech.get(\'overlay_path\')}`",\n        f"- measurements_count: `{len(tech.get(\'measurements\', []))}`", \'\',\n        \'## Decisión del agente\', \'\',\n        f"- priority: `{dec.get(\'priority\')}`", f"- status: `{dec.get(\'status\')}`", f"- flags: `{dec.get(\'flags\')}`", f"- action_recommended: {dec.get(\'action_recommended\')}", \'\',\n        \'La salida es un artefacto técnico de apoyo. Requiere revisión profesional obligatoria y no constituye diagnóstico clínico autónomo.\'\n    ]\n    path = output_dir / f\'{run_id}.md\'\n    path.write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    return path\n',
    PACKAGE_ROOT / 'service_endpoints_runtime.py': 'from __future__ import annotations\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\nimport json, uuid, datetime\nimport numpy as np\nfrom .agent_policy import evaluate_agent_policy\nfrom .case_reporting import build_case_report, write_json_report, write_markdown_report\ntry:\n    from PIL import Image\nexcept Exception:\n    Image = None\n\ndef repo_root() -> Path:\n    return Path(__file__).resolve().parents[2]\n\ndef pfi_root() -> Path:\n    return repo_root().parent\n\ndef results_root() -> Path:\n    return pfi_root() / \'results\' / \'PF012_PF019_service_endpoints_agent\' / \'service_runs\'\n\ndef load_model_registry() -> Dict[str, Any]:\n    path = repo_root() / \'config\' / \'model_registry_final.json\'\n    if not path.exists(): raise FileNotFoundError(str(path))\n    data = json.loads(path.read_text(encoding=\'utf-8\'))\n    if isinstance(data, dict) and isinstance(data.get(\'models\'), dict): return data[\'models\']\n    if isinstance(data, dict) and isinstance(data.get(\'models\'), list): return {m.get(\'model_key\', f\'model_{i}\'): m for i,m in enumerate(data[\'models\'])}\n    if isinstance(data, list): return {m.get(\'model_key\', f\'model_{i}\'): m for i,m in enumerate(data)}\n    return data if isinstance(data, dict) else {}\n\ndef models_payload():\n    reg=load_model_registry(); items=[]\n    for key, meta in reg.items():\n        meta = meta if isinstance(meta, dict) else {\'value\': meta}\n        path = meta.get(\'final_path\') or meta.get(\'checkpoint_path\') or meta.get(\'path\')\n        items.append({\'model_key\':key,\'plane\':meta.get(\'plane\'),\'dataset_key\':meta.get(\'dataset_key\'),\'source_stage\':meta.get(\'source_stage\'),\'final_path\':path,\'exists\':bool(path and Path(path).exists()),\'sha256\':meta.get(\'sha256\')})\n    return {\'models\':items,\'count\':len(items),\'human_review_required\':True}\n\ndef default_model_for_plane(plane: str) -> str:\n    if plane == \'axial\': return \'axial_t2_alkafri\'\n    if plane == \'sagittal\': return \'sagittal_spider\'\n    raise ValueError("plane must be \'sagittal\' or \'axial\'")\n\ndef synthetic_image_and_mask(plane: str, size=256):\n    y,x=np.mgrid[0:size,0:size]\n    img=(0.15+0.55*np.exp(-((x-size/2)**2+(y-size/2)**2)/(2*(size/3)**2))).astype(np.float32)\n    mask=np.zeros((size,size),dtype=np.uint8)\n    if plane==\'sagittal\':\n        mask[(x>82)&(x<142)&(y>35)&(y<220)]=1\n        for cy in [65,105,145,185]: mask[(x>145)&(x<188)&(np.abs(y-cy)<10)]=3\n        mask[(x>195)&(x<216)&(y>35)&(y<220)]=2\n    else:\n        rr=(x-size/2)**2+(y-size/2)**2\n        mask[rr<(size*0.22)**2]=1\n        mask[((x-size/2)**2+(y-size*0.42)**2)<(size*0.10)**2]=2\n        mask[((x-size/2)**2+(y-size*0.63)**2)<(size*0.12)**2]=3\n    return img,mask\n\ndef load_simple_file(file_path: str):\n    path=Path(file_path); warnings=[]\n    if not path.exists(): raise FileNotFoundError(file_path)\n    suf=path.suffix.lower()\n    if suf==\'.npy\':\n        arr=np.load(path); arr=arr[arr.shape[0]//2] if arr.ndim==3 else arr\n        return arr.astype(np.float32), np.zeros(arr.shape[:2], dtype=np.uint8), [\'NPY cargado sin máscara; se requiere inferencia real para máscara no vacía.\']\n    if suf in {\'.png\',\'.jpg\',\'.jpeg\',\'.bmp\'}:\n        if Image is None: raise RuntimeError(\'Pillow no instalado\')\n        img=np.asarray(Image.open(path).convert(\'L\')).astype(np.float32)/255.0\n        return img, np.zeros(img.shape, dtype=np.uint8), [\'Imagen simple cargada sin máscara; máscara vacía para contrato.\']\n    raise ValueError(f\'Tipo de archivo no soportado en smoke actual: {suf}\')\n\ndef measure_mask(mask, spacing=(1.0,1.0)):\n    out=[]; pix=float(spacing[0]*spacing[1])\n    for cls in sorted([int(c) for c in np.unique(mask) if int(c)!=0]):\n        ys,xs=np.where(mask==cls)\n        out.append({\'class_id\':cls,\'area_px\':int(len(xs)),\'area_units2\':float(len(xs)*pix),\'bbox_xmin\':int(xs.min()),\'bbox_xmax\':int(xs.max()),\'bbox_ymin\':int(ys.min()),\'bbox_ymax\':int(ys.max()),\'centroid_x\':float(xs.mean()),\'centroid_y\':float(ys.mean())})\n    return out\n\ndef save_overlay(image, mask, output_path: Path):\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    img=image.astype(np.float32); img=img-np.nanmin(img); img=img/(float(np.nanmax(img)) or 1.0)\n    if Image is None:\n        np.save(str(output_path.with_suffix(\'.npy\')), np.stack([img, mask.astype(np.float32)])); return output_path.with_suffix(\'.npy\')\n    rgb=np.stack([img,img,img],axis=-1); out=rgb.copy(); alpha=0.4\n    pal={1:np.array([1.0,0.2,0.2]),2:np.array([0.2,1.0,0.2]),3:np.array([0.2,0.4,1.0]),4:np.array([1.0,0.8,0.2])}\n    for cls,col in pal.items():\n        m=mask==cls; out[m]=(1-alpha)*out[m]+alpha*col\n    Image.fromarray((np.clip(out,0,1)*255).astype(np.uint8)).save(output_path)\n    return output_path\n\ndef run_service_pipeline(payload: Dict[str, Any]):\n    plane=str(payload.get(\'plane\') or \'\').lower()\n    if plane not in {\'sagittal\',\'axial\'}: raise ValueError("plane must be \'sagittal\' or \'axial\'")\n    model_key=payload.get(\'model_key\') or default_model_for_plane(plane)\n    if model_key not in load_model_registry(): raise ValueError(f\'model_key not found: {model_key}\')\n    run_id=payload.get(\'run_id\') or f"run_{datetime.datetime.now().strftime(\'%Y%m%d_%H%M%S\')}_{uuid.uuid4().hex[:8]}"\n    synthetic=bool(payload.get(\'synthetic\',False)); file_path=payload.get(\'file_path\'); warnings=[]\n    if synthetic or not file_path:\n        image,mask=synthetic_image_and_mask(plane)\n        if not synthetic: warnings.append(\'No se recibió file_path; se ejecutó synthetic smoke para validar contrato.\')\n    else:\n        image,mask,warnings=load_simple_file(file_path)\n    present=[int(c) for c in np.unique(mask) if int(c)!=0]\n    overlay=save_overlay(image, mask, results_root()/\'overlays\'/f\'{run_id}_{plane}_overlay.png\')\n    tech={\'run_id\':run_id,\'case_id\':payload.get(\'case_id\',run_id),\'plane\':plane,\'model_key\':model_key,\'synthetic\':synthetic,\'file_path\':file_path,\'overlay_path\':str(overlay),\'foreground_ratio\':float((mask>0).mean()),\'present_classes\':present,\'measurements\':measure_mask(mask),\'warnings\':warnings,\'errors\':[],\'created_at\':datetime.datetime.now().replace(microsecond=0).isoformat()}\n    decision=evaluate_agent_policy(tech)\n    report=build_case_report({**payload,\'case_id\':tech[\'case_id\'],\'plane\':plane,\'model_key\':model_key}, tech, decision)\n    report_dir=results_root()/\'case_reports\'\n    tech[\'case_report_json\']=str(write_json_report(report, report_dir)); tech[\'case_report_md\']=str(write_markdown_report(report, report_dir))\n    return {\'technical_result\':tech,\'agent_decision\':decision,\'case_report\':report}\n\ndef load_case_report(run_id: str):\n    path=results_root()/\'case_reports\'/f\'{run_id}.json\'\n    if not path.exists(): raise FileNotFoundError(f\'case report not found: {run_id}\')\n    return json.loads(path.read_text(encoding=\'utf-8\'))\n',
    PACKAGE_ROOT / 'api.py': "from __future__ import annotations\nfrom typing import Any, Dict, Optional\nimport traceback\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nfrom .service_endpoints_runtime import models_payload, run_service_pipeline, load_case_report\nfrom .agent_policy import run_agent_regression\n\napp = FastAPI(title='PFI Lumbar MRI AI Service', version='0.2.0-pf012-pf019', description='Servicio técnico de apoyo con revisión profesional obligatoria.')\n\nclass InferenceRequest(BaseModel):\n    case_id: Optional[str] = None\n    plane: str = Field(description='sagittal o axial')\n    model_key: Optional[str] = None\n    file_path: Optional[str] = None\n    synthetic: bool = False\n\nclass PipelineRunRequest(BaseModel):\n    case_id: Optional[str] = None\n    plane: str\n    model_key: Optional[str] = None\n    file_path: Optional[str] = None\n    synthetic: bool = False\n\ndef _dump(model):\n    return model.model_dump() if hasattr(model, 'model_dump') else model.dict()\n\ndef _safe(payload: Dict[str, Any]):\n    try: return run_service_pipeline(payload)\n    except Exception as exc:\n        raise HTTPException(status_code=422, detail={'error':str(exc),'traceback_tail':traceback.format_exc().splitlines()[-5:],'human_review_required':True,'not_clinical_diagnosis':True})\n\n@app.get('/health')\ndef health():\n    return {'status':'ok','service':'pfi_ai_service','version':'0.2.0-pf012-pf019','human_review_required':True,'not_clinical_diagnosis':True,'not_real_3d_reconstruction':True}\n\n@app.get('/models')\ndef models():\n    try: return models_payload()\n    except Exception as exc: raise HTTPException(status_code=500, detail=str(exc))\n\n@app.post('/inference/sagittal')\ndef inference_sagittal(req: InferenceRequest):\n    p=_dump(req); p['plane']='sagittal'; return _safe(p)\n\n@app.post('/inference/axial')\ndef inference_axial(req: InferenceRequest):\n    p=_dump(req); p['plane']='axial'; return _safe(p)\n\n@app.post('/pipeline/run')\ndef pipeline_run(req: PipelineRunRequest):\n    return _safe(_dump(req))\n\n@app.get('/agent/regression-test')\ndef agent_regression_test():\n    return run_agent_regression()\n\n@app.get('/agent/report/{run_id}')\ndef get_agent_report(run_id: str):\n    try: return load_case_report(run_id)\n    except Exception as exc: raise HTTPException(status_code=404, detail=str(exc))\n\n@app.get('/agent/worklist')\ndef agent_worklist():\n    return {'items':[],'count':0,'source':'live_service_placeholder','note':'La worklist persistente se integrará en Spring Boot en PF-020 a PF-025.','human_review_required':True}\n",
    AI_SERVICE_ROOT / 'Dockerfile': 'FROM python:3.11-slim\nWORKDIR /app\nENV PYTHONDONTWRITEBYTECODE=1\nENV PYTHONUNBUFFERED=1\nENV PYTHONPATH=/app/ai_service\nCOPY ai_service /app/ai_service\nCOPY config /app/config\nRUN pip install --no-cache-dir fastapi uvicorn pydantic numpy pillow python-multipart\nEXPOSE 8000\nCMD ["uvicorn", "pfi_ai_service.api:app", "--host", "0.0.0.0", "--port", "8000"]\n',
}
for path, content in files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
    print('Wrote:', path)


Wrote: /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/agent_policy.py
Wrote: /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/case_reporting.py
Wrote: /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/service_endpoints_runtime.py
Wrote: /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/api.py
Wrote: /content/drive/MyDrive/PFI_MVP/repo/ai_service/Dockerfile


In [4]:
import pandas as pd
files = [
    REPO_ROOT/'ai_service/pfi_ai_service/agent_policy.py',
    REPO_ROOT/'ai_service/pfi_ai_service/case_reporting.py',
    REPO_ROOT/'ai_service/pfi_ai_service/service_endpoints_runtime.py',
    REPO_ROOT/'ai_service/pfi_ai_service/api.py',
    REPO_ROOT/'ai_service/Dockerfile',
]
module_inventory_df = pd.DataFrame([{'relative_path':str(p.relative_to(REPO_ROOT)), 'exists':p.exists(), 'size_bytes':p.stat().st_size if p.exists() else 0} for p in files])
module_inventory_csv = RESULTS_ROOT/'PF012_PF019_generated_files_inventory.csv'
module_inventory_df.to_csv(module_inventory_csv, index=False)
display(module_inventory_df)
print('Module inventory:', module_inventory_csv)


,relative_path,exists,size_bytes
0,ai_service/pfi_ai_service/agent_policy.py,True,4510
1,ai_service/pfi_ai_service/case_reporting.py,True,2600
2,ai_service/pfi_ai_service/service_endpoints_ru...,True,6759
3,ai_service/pfi_ai_service/api.py,True,2541
4,ai_service/Dockerfile,True,354


Module inventory: /content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_generated_files_inventory.csv


In [5]:
# FastAPI smoke test
sys.path.insert(0, str(REPO_ROOT/'ai_service'))
import importlib
import pfi_ai_service.api as api_module
importlib.reload(api_module)
from fastapi.testclient import TestClient
client = TestClient(api_module.app)
endpoint_results=[]

def add_result(name, resp, ok_extra=True):
    try: detail = resp.json()
    except Exception: detail = resp.text
    endpoint_results.append({'endpoint':name, 'status_code':resp.status_code, 'ok':resp.status_code==200 and ok_extra, 'detail':detail})

r=client.get('/health'); add_result('GET /health', r)
r=client.get('/models'); add_result('GET /models', r, r.status_code==200 and r.json().get('count',0)>=2)
r_sag=client.post('/inference/sagittal', json={'case_id':'SMOKE_SAG_001','plane':'sagittal','synthetic':True}); add_result('POST /inference/sagittal', r_sag)
r_ax=client.post('/inference/axial', json={'case_id':'SMOKE_AX_001','plane':'axial','synthetic':True}); add_result('POST /inference/axial', r_ax)
r_pipe=client.post('/pipeline/run', json={'case_id':'SMOKE_PIPE_001','plane':'sagittal','synthetic':True}); add_result('POST /pipeline/run', r_pipe)
run_id = r_pipe.json().get('technical_result',{}).get('run_id') if r_pipe.status_code==200 else None
if run_id:
    r_rep=client.get(f'/agent/report/{run_id}'); add_result('GET /agent/report/{run_id}', r_rep)
else:
    endpoint_results.append({'endpoint':'GET /agent/report/{run_id}','status_code':0,'ok':False,'detail':'no run_id'})
r_reg=client.get('/agent/regression-test'); add_result('GET /agent/regression-test', r_reg, r_reg.status_code==200 and r_reg.json().get('all_ok') is True)
endpoint_summary_df = pd.DataFrame([{'endpoint':x['endpoint'],'status_code':x['status_code'],'ok':x['ok']} for x in endpoint_results])
endpoint_summary_csv = RESULTS_ROOT/'PF012_PF019_endpoint_smoke_summary.csv'
endpoint_summary_df.to_csv(endpoint_summary_csv, index=False)
display(endpoint_summary_df)
print('All endpoint smoke OK:', bool(endpoint_summary_df['ok'].all()))


,endpoint,status_code,ok
0,GET /health,200,True
1,GET /models,200,True
2,POST /inference/sagittal,200,True
3,POST /inference/axial,200,True
4,POST /pipeline/run,200,True
5,GET /agent/report/{run_id},200,True
6,GET /agent/regression-test,200,True


All endpoint smoke OK: True


In [6]:
checks=[]
checks.append({'check':'agent_policy_written','ok':(PACKAGE_ROOT/'agent_policy.py').exists(),'detail':str(PACKAGE_ROOT/'agent_policy.py')})
checks.append({'check':'case_reporting_written','ok':(PACKAGE_ROOT/'case_reporting.py').exists(),'detail':str(PACKAGE_ROOT/'case_reporting.py')})
checks.append({'check':'service_runtime_written','ok':(PACKAGE_ROOT/'service_endpoints_runtime.py').exists(),'detail':str(PACKAGE_ROOT/'service_endpoints_runtime.py')})
checks.append({'check':'api_updated','ok':(PACKAGE_ROOT/'api.py').exists(),'detail':str(PACKAGE_ROOT/'api.py')})
checks.append({'check':'dockerfile_written','ok':(AI_SERVICE_ROOT/'Dockerfile').exists(),'detail':str(AI_SERVICE_ROOT/'Dockerfile')})
checks.append({'check':'all_endpoint_smoke_ok','ok':bool(endpoint_summary_df['ok'].all()),'detail':str(endpoint_summary_df.to_dict(orient='records'))})
policy_ok=False
if r_pipe.status_code==200:
    data=r_pipe.json(); policy_ok=data.get('agent_decision',{}).get('human_review_required') is True and data.get('agent_decision',{}).get('not_clinical_diagnosis') is True
checks.append({'check':'human_review_policy_in_response','ok':policy_ok,'detail':'agent_decision includes policy flags'})
checks_df=pd.DataFrame(checks)
checks_csv=RESULTS_ROOT/'PF012_PF019_checks.csv'
checks_df.to_csv(checks_csv,index=False)
display(checks_df)
print('Checks:', checks_csv)
print('All OK:', bool(checks_df['ok'].all()))
smoke_report={'endpoint_results':endpoint_results,'run_id':run_id,'all_endpoint_smoke_ok':bool(endpoint_summary_df['ok'].all())}
smoke_report_json=RESULTS_ROOT/'PF012_PF019_endpoint_smoke_report.json'
smoke_report_json.write_text(json.dumps(smoke_report, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
report={
    'notebook':'31_PF012_PF019_service_endpoints_agent',
    'goal':'harden agent policy and expose product inference modules through FastAPI endpoints',
    'timestamp':datetime.datetime.now().replace(microsecond=0).isoformat(),
    'outputs':{
        'generated_files_inventory_csv':str(module_inventory_csv),
        'endpoint_smoke_summary_csv':str(endpoint_summary_csv),
        'endpoint_smoke_report_json':str(smoke_report_json),
        'checks_csv':str(checks_csv),
        'docs_repo':str(DOCS_ROOT/'PF012_PF019_service_endpoints_agent.md'),
        'dockerfile':str(AI_SERVICE_ROOT/'Dockerfile'),
    },
    'summary':{
        'generated_files':int(module_inventory_df['exists'].sum()),
        'endpoint_smoke_tests':int(len(endpoint_summary_df)),
        'all_endpoint_smoke_ok':bool(endpoint_summary_df['ok'].all()),
        'agent_regression_ok':bool(r_reg.status_code==200 and r_reg.json().get('all_ok') is True),
        'run_id_smoke_pipeline':run_id,
    },
    'methodological_policy':{
        'human_review_required':True,
        'not_clinical_diagnosis':True,
        'not_real_3d_reconstruction':True,
        'product_scope':'MVP técnico con endpoints FastAPI de inferencia 2D multiplanar y revisión profesional',
    },
    'decision':'PF012_PF019_service_endpoints_ready_for_spring_boot_backend',
}
report_json=RESULTS_ROOT/'PF012_PF019_report.json'
report_json.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
md_text=f"""# PF-012 a PF-019 - Servicio Python, endpoints y agente

## Resultado
Se actualizaron los módulos del servicio Python para exponer endpoints FastAPI.

## Endpoints validados
- GET /health
- GET /models
- POST /inference/sagittal
- POST /inference/axial
- POST /pipeline/run
- GET /agent/report/{{run_id}}
- GET /agent/regression-test

## Decision
`{report['decision']}`

La salida es técnica y asistiva. Requiere revisión profesional obligatoria y no constituye diagnóstico clínico autónomo.
"""
(DOCS_ROOT/'PF012_PF019_service_endpoints_agent.md').write_text(md_text, encoding='utf-8')
closure=f"""# Cierre PF-012 a PF-019 - Endpoints del servicio Python y agente

Estado: cerrado.

Decision: `{report['decision']}`

Resumen:
- archivos generados: {report['summary']['generated_files']}
- endpoints testeados: {report['summary']['endpoint_smoke_tests']}
- all_endpoint_smoke_ok: {report['summary']['all_endpoint_smoke_ok']}
- agent_regression_ok: {report['summary']['agent_regression_ok']}
- run_id smoke pipeline: `{report['summary']['run_id_smoke_pipeline']}`

Próximo bloque: PF-020 a PF-025, backend Spring Boot real, DTOs definitivos y consumo del servicio Python.
"""
(BACKLOG_ROOT/'PF012_PF019_resultados_cierre.md').write_text(closure, encoding='utf-8')
print('Smoke report:', smoke_report_json)
print('Report:', report_json)
print('Docs:', DOCS_ROOT/'PF012_PF019_service_endpoints_agent.md')
print('Closure:', BACKLOG_ROOT/'PF012_PF019_resultados_cierre.md')
print(json.dumps(report, indent=2, ensure_ascii=False))


,check,ok,detail
0,agent_policy_written,True,/content/drive/MyDrive/PFI_MVP/repo/ai_service...
1,case_reporting_written,True,/content/drive/MyDrive/PFI_MVP/repo/ai_service...
2,service_runtime_written,True,/content/drive/MyDrive/PFI_MVP/repo/ai_service...
3,api_updated,True,/content/drive/MyDrive/PFI_MVP/repo/ai_service...
4,dockerfile_written,True,/content/drive/MyDrive/PFI_MVP/repo/ai_service...
5,all_endpoint_smoke_ok,True,"[{'endpoint': 'GET /health', 'status_code': 20..."
6,human_review_policy_in_response,True,agent_decision includes policy flags


Checks: /content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_checks.csv
All OK: True
Smoke report: /content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_endpoint_smoke_report.json
Report: /content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_report.json
Docs: /content/drive/MyDrive/PFI_MVP/repo/docs/PF012_PF019_service_endpoints_agent.md
Closure: /content/drive/MyDrive/PFI_MVP/repo/backlogProducto/PF012_PF019_resultados_cierre.md
{
  "notebook": "31_PF012_PF019_service_endpoints_agent",
  "goal": "harden agent policy and expose product inference modules through FastAPI endpoints",
  "timestamp": "2026-06-30T22:31:16",
  "outputs": {
    "generated_files_inventory_csv": "/content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_generated_files_inventory.csv",
    "endpoint_smoke_summary_csv": "/content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoint

## Commit sugerido

```bash
%cd /content/drive/MyDrive/PFI_MVP/repo
!git status
!git add ai_service/pfi_ai_service/agent_policy.py ai_service/pfi_ai_service/case_reporting.py ai_service/pfi_ai_service/service_endpoints_runtime.py ai_service/pfi_ai_service/api.py ai_service/Dockerfile docs/PF012_PF019_service_endpoints_agent.md backlogProducto/PF012_PF019_plan.md backlogProducto/PF012_PF019_resultados_cierre.md notebooks/31_PF012_PF019_service_endpoints_agent.ipynb
!git commit -m "Add PF012 PF019 FastAPI service endpoints and agent policy"
!git push
```
